In [ ]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from typing import TypedDict, Literal, Annotated
from langchain_core.messages import SystemMessage, HumanMessage, BaseMessage
from langgraph.checkpoint.memory import MemorySaver

In [ ]:
from langgraph.graph.message import add_messages

class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [ ]:
llm = ChatOpenAI()
def chat_node(state: ChatState):
    messages = state['messages']

    response = llm.invoke(messages)

    return {'messages': [response]}

In [ ]:
checkpointer = MemorySaver()

graph = StateGraph(ChatState)

graph.add_node('chat_node', chat_node)

graph.add_edge(START, 'chat_node')
graph.add_edge('chat_node', END)

chatbot = graph.compile(checkpointer=checkpointer)

In [ ]:
thread_id = '1'

while True:
    user_message = input('type here: ')

    print('user:', user_message)

    if user_message.strip().lower() in ['exit', 'quit', 'bye']:
        break

    config = {'configurable': {'thread_id': thread_id}}
        
    # invoking the graph over and over again in the loop uses new states eveytime and thats why, the llm cant remember previous chat    
    response = chatbot.invoke({'messages': {HumanMessage(content=user_message)}}, config = config)

    print('AI:', response['messages'][-1].content)